## Compare RT-DETR models on DS_6 videos (C1..C8)

Runs inference on the 8 DS_6 videos with two RT-DETR weights (pulled from Hugging Face) and dumps per-video JSONs for both models:

- **Custom (Ulysse)**: `estefoucher/tell-tale-detector` → `weights/custom_ulysse.pt`.
- **After augmentation**: `estefoucher/tell-tale-detector` → `runs/finetune_rtdetr_fused_run_17_epoch.zip` (extracts `weights/last.pt`).

Drive is mounted FIRST so the session can run unattended. All outputs are saved to `MyDrive/SailCVExports/ds6_compare_runs/<timestamp>/`.

## 1. Mount Google Drive (first — so you can sleep)

In [ ]:
from datetime import datetime
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
DRIVE_OUT_BASE = None

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_OUT_BASE = Path('/content/drive/MyDrive/SailCVExports/ds6_compare_runs') / RUN_STAMP
    DRIVE_OUT_BASE.mkdir(parents=True, exist_ok=True)
    print('Drive mounted. Output folder:', DRIVE_OUT_BASE)
else:
    print('Not in Colab: Drive export disabled.')

## 2. Check GPU + install dependencies

In [ ]:
!nvidia-smi || true

In [ ]:
!pip install -q ultralytics huggingface_hub opencv-python-headless tqdm loguru pyyaml pydantic
from IPython import display
display.clear_output()
print('Dependencies installed.')

## 3. Clone the Sail-CV repo (brings the DS_6 videos)

In [ ]:
import os, sys
from pathlib import Path

HOME = os.getcwd()
print('HOME =', HOME)

!git clone --depth 1 https://github.com/estebanfoucher/sail-CV.git sailcv_repo || (cd sailcv_repo && git pull)

repo_root = Path(HOME) / 'sailcv_repo'
print('Repo root:', repo_root)

## 4. Download both weights from Hugging Face

- `weight_custom_ulysse_path`: `weights/custom_ulysse.pt`.
- `weight_after_path`: `runs/finetune_rtdetr_fused_run_17_epoch.zip` → `weights/last.pt`.

In [ ]:
import shutil
import zipfile
from pathlib import Path

from huggingface_hub import hf_hub_download

HF_REPO = 'estefoucher/tell-tale-detector'
CUSTOM_ULYSSE_FILE = 'weights/custom_ulysse.pt'
AFTER_FILE = 'runs/finetune_rtdetr_fused_run_17_epoch.zip'

weights_dir = Path(HOME) / 'weights'
weights_dir.mkdir(parents=True, exist_ok=True)

weight_custom_ulysse_path = Path(hf_hub_download(
    repo_id=HF_REPO,
    filename=CUSTOM_ULYSSE_FILE,
    repo_type='model',
    local_dir=str(weights_dir / 'custom_ulysse'),
    local_dir_use_symlinks=False,
))
print('Custom Ulysse weight:', weight_custom_ulysse_path,
      f'({weight_custom_ulysse_path.stat().st_size / 1e6:.1f} MB)')

run_zip_path = Path(hf_hub_download(
    repo_id=HF_REPO,
    filename=AFTER_FILE,
    repo_type='model',
    local_dir=str(weights_dir / 'after_zip'),
    local_dir_use_symlinks=False,
))
print('After run zip:', run_zip_path, f'({run_zip_path.stat().st_size / 1e6:.1f} MB)')

after_extract_dir = weights_dir / 'after_extracted'
if after_extract_dir.exists():
    shutil.rmtree(after_extract_dir)
after_extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(run_zip_path, 'r') as zf:
    zf.extractall(after_extract_dir)

last_candidates = list(after_extract_dir.rglob('weights/last.pt'))
if not last_candidates:
    raise FileNotFoundError(f'weights/last.pt not found inside {run_zip_path}')
weight_after_path = last_candidates[0]
print('After last.pt:', weight_after_path)

## 5. Locate the 8 DS_6 videos (C1..C8)

Videos live in the cloned repo at `sailcv_repo/assets/tracking/DS_6/C{1..8}.mp4` — no manual upload needed.

In [ ]:
from pathlib import Path

ds6_dir = repo_root / 'assets' / 'tracking' / 'DS_6'
if not ds6_dir.is_dir():
    raise FileNotFoundError(
        f'DS_6 folder not found at {ds6_dir}. Did `git clone` fail or the repo layout changed?'
    )

videos_to_process = []
missing = []
for i in range(1, 9):
    tag = f'C{i}'
    video_path = ds6_dir / f'{tag}.mp4'
    if video_path.is_file():
        videos_to_process.append((tag, video_path.resolve()))
    else:
        missing.append(tag)

if missing:
    print('WARNING: missing DS_6 videos:', missing)

if not videos_to_process:
    raise FileNotFoundError(f'No C1..C8.mp4 found under {ds6_dir}.')

print(f'Found {len(videos_to_process)} DS_6 videos:')
for tag, p in videos_to_process:
    size_mb = p.stat().st_size / 1e6
    print(f'  {tag}: {p}  ({size_mb:.1f} MB)')

## 6. Inference helper (uses repo `Detector` class)

Uses the Sail-CV `Detector` / `Model` classes (`src/tracking/detector.py`) instead of a custom Ultralytics wrapper. This gives the exact same code path as the production pipeline.

Ultralytics has a known issue on Colab (recent versions) when `.half()` is called before the lazy `.fuse()` in `predict()` → `RuntimeError: expected mat1 and mat2 to have the same dtype`. The workaround here is to **force FP32** after `Detector` init (reverts the `.half()` done inside `Model.__init__`). This keeps inference correct; on Colab GPUs (T4/L4/A100) FP32 RTDETR-L is fast enough for the 8 DS_6 videos.

JSON schema unchanged: `{"<frame>": [{"bbox":{"xyxy":{...}, ...}, "confidence":..., "class_id":..., "class_name":..., ...}]}`, compatible with the repo's `FakeModel`.

In [ ]:
import json
import sys
import time
from pathlib import Path

import cv2
import torch
from tqdm import tqdm

tracking_src = (repo_root / 'src' / 'tracking').resolve()
tracking_src_str = str(tracking_src)
tracking_models_str = str(tracking_src / 'models')

sys.path[:] = [p for p in sys.path if Path(p).resolve() != Path(tracking_models_str).resolve()]

if tracking_src_str not in sys.path:
    sys.path.insert(0, tracking_src_str)

for m in list(sys.modules):
    if m == 'detector' or m == 'models' or m.startswith('models.') or m.startswith('detector.'):
        sys.modules.pop(m, None)

from detector import Detector  # noqa: E402
from models import Image, ModelSpecs  # noqa: E402

import detector as _detector_mod
import models as _models_mod
assert Path(_detector_mod.__file__).resolve() == (tracking_src / 'detector.py'), (
    f'Wrong detector.py loaded: {_detector_mod.__file__}'
)
assert Path(_models_mod.__file__).resolve().parent == (tracking_src / 'models'), (
    f'Wrong models package loaded: {_models_mod.__file__}'
)
print('Loaded repo Detector from:', _detector_mod.__file__)
print('Loaded repo models package from:', _models_mod.__file__)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

CONF = 0.25
IOU = 0.5
FORCE_FP32 = True  # workaround for Ultralytics fuse+half dtype bug on Colab


def build_detector(weight_path: Path) -> Detector:
    """Build the repo's Detector. Force FP32 after init to avoid Ultralytics fuse+half bug."""
    specs = ModelSpecs(model_path=Path(weight_path), architecture='rt-detr')
    detector = Detector(specs=specs)
    _ = detector.model  # triggers lazy Model() init (which halves on cuda)
    if FORCE_FP32 and DEVICE == 'cuda':
        try:
            detector._model.model.float()
            print(f'  Forced FP32 for {weight_path.name}.')
        except Exception as exc:
            print(f'  Could not force FP32 ({exc}); will rely on default dtype.')
    try:
        ultra_model = detector._model.model
        ultra_model.overrides['conf'] = CONF
        ultra_model.overrides['iou'] = IOU
    except Exception:
        pass
    return detector


def _detection_record(det, w, h):
    xyxy = det.bbox.xyxy
    x1, y1, x2, y2 = int(xyxy.x1), int(xyxy.y1), int(xyxy.x2), int(xyxy.y2)
    bw = max(0, x2 - x1)
    bh = max(0, y2 - y1)
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    return {
        'bbox': {
            'xyxy': {'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2},
            'xywh': {'cx': cx, 'cy': cy, 'w': float(bw), 'h': float(bh)},
            'xywhn': {
                'cx': cx / w if w else 0.0,
                'cy': cy / h if h else 0.0,
                'w': bw / w if w else 0.0,
                'h': bh / h if h else 0.0,
            },
            'width': int(bw),
            'height': int(bh),
            'area': int(bw * bh),
        },
        'confidence': float(det.confidence),
        'class_id': int(det.class_id),
        'image_width': w,
        'image_height': h,
    }


def run_inference(detector: Detector, video_path: Path, json_out: Path) -> dict:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open {video_path}')
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    try:
        names = dict(getattr(detector._model.model, 'names', {}) or {})
    except Exception:
        names = {}

    results_dict: dict[str, list] = {}
    total_dets = 0
    frame_idx = 0
    t_start = time.perf_counter()

    pbar = tqdm(total=total or None, desc=video_path.name)
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        image = Image(image=frame, rgb_bgr='BGR')
        detections = detector.detect(image)

        frame_dets = []
        for det in detections:
            record = _detection_record(det, width, height)
            record['class_name'] = names.get(record['class_id'], str(record['class_id']))
            frame_dets.append(record)

        results_dict[str(frame_idx)] = frame_dets
        total_dets += len(frame_dets)
        frame_idx += 1
        pbar.update(1)
    pbar.close()
    cap.release()

    elapsed = time.perf_counter() - t_start
    meta = {
        'video': str(video_path),
        'width': width,
        'height': height,
        'fps': fps,
        'frames_processed': frame_idx,
        'total_detections': total_dets,
        'conf': CONF,
        'iou': IOU,
        'device': DEVICE,
        'force_fp32': FORCE_FP32,
        'class_names': names,
        'tracking': False,
        'detector_class': 'src.tracking.detector.Detector',
        'elapsed_sec': round(elapsed, 2),
    }

    json_out.parent.mkdir(parents=True, exist_ok=True)
    with json_out.open('w') as f:
        json.dump(results_dict, f)
    meta_out = json_out.with_suffix('.meta.json')
    with meta_out.open('w') as f:
        json.dump(meta, f, indent=2)

    print(f'  {json_out.name}: {frame_idx} frames, {total_dets} detections in {elapsed:.1f}s')
    return meta

## 7. Run inference with both models on all 8 DS videos

Produces for every video and every model:

- `<model>/<Ci>_raw_detection.json`
- `<model>/<Ci>_raw_detection.meta.json`

In [ ]:
from pathlib import Path

out_root = Path(HOME) / 'ds6_compare_out'
out_root.mkdir(parents=True, exist_ok=True)

MODEL_CONFIGS = [
    ('custom_ulysse', weight_custom_ulysse_path),
    ('after_finetune_fused_17ep_last', weight_after_path),
]

summary = {'run_stamp': RUN_STAMP, 'videos': {}, 'models': {}}

for model_tag, weight_path in MODEL_CONFIGS:
    print(f'\n=== Model: {model_tag} ===')
    print('Weight:', weight_path)
    detector = build_detector(Path(weight_path))
    model_out = out_root / model_tag
    model_out.mkdir(parents=True, exist_ok=True)

    model_summary = {}
    for tag, video_path in videos_to_process:
        json_out = model_out / f'{tag}_raw_detection.json'
        meta = run_inference(detector, Path(video_path), json_out)
        model_summary[tag] = {
            'total_detections': meta['total_detections'],
            'frames_processed': meta['frames_processed'],
            'elapsed_sec': meta['elapsed_sec'],
            'json': str(json_out.relative_to(out_root)),
        }
        summary['videos'].setdefault(tag, {'path': str(video_path)})
    summary['models'][model_tag] = {
        'weight': str(weight_path),
        'results': model_summary,
    }

    del detector
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

summary_path = out_root / 'compare_summary.json'
with summary_path.open('w') as f:
    json.dump(summary, f, indent=2)
print('\nSummary written:', summary_path)

## 8. Save all JSONs to Google Drive + zip backup

In [ ]:
import shutil
from pathlib import Path

zip_basename = f'ds6_compare_{RUN_STAMP}'
zip_path = Path(HOME) / f'{zip_basename}.zip'
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(
    base_name=str(Path(HOME) / zip_basename),
    format='zip',
    root_dir=str(out_root.parent),
    base_dir=out_root.name,
)
print(f'Local zip: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)')

if DRIVE_OUT_BASE is not None:
    drive_tree = DRIVE_OUT_BASE / out_root.name
    if drive_tree.exists():
        shutil.rmtree(drive_tree)
    shutil.copytree(out_root, drive_tree)
    print('Drive folder copy:', drive_tree)

    drive_zip = DRIVE_OUT_BASE / zip_path.name
    shutil.copy2(zip_path, drive_zip)
    print('Drive zip copy:', drive_zip)

## 9. Quick comparison (detections per video, per model)

In [ ]:
print(f"{'Video':<6} | " + ' | '.join(f'{m:<40}' for m, _ in MODEL_CONFIGS))
print('-' * (8 + sum(43 for _ in MODEL_CONFIGS)))

all_tags = sorted({t for t, _ in videos_to_process})
for tag in all_tags:
    row_vals = []
    for model_tag, _ in MODEL_CONFIGS:
        info = summary['models'][model_tag]['results'].get(tag)
        if info is None:
            row_vals.append('n/a')
        else:
            row_vals.append(
                f"{info['total_detections']:>6} dets / {info['frames_processed']:>5} fr"
            )
    print(f'{tag:<6} | ' + ' | '.join(f'{v:<40}' for v in row_vals))